# Lab: adaptation on synthetic worlds

The **Bayesian update** box from week 2 learns the world model's errors. This week you run it
with library BLR and GP on synthetic worlds, where the true error is known and the world changes
at step 100.

Fill in every cell marked **TODO**. Needs `pip install scikit-learn matplotlib`; run from the repo root.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../..') if os.getcwd().endswith('tracking') else os.getcwd())

from sklearn.linear_model import BayesianRidge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel, Matern

from labs.tracking.bench import plot
from labs.tracking.models import Frozen, Online

## 1. The worlds

- **Gray dots:** what we observe (the world model's error, plus sensor noise).
- **Black line:** the true error.
- **Tasks:** `hover` (drone-like), `swing` (pendulum-like), `push`.

In [ ]:
plot({}, 'hover', seed=4)
plot({}, 'swing', seed=1)
plot({}, 'hover', seed=4, sensor=0.1);

## 2. BLR

`Online(regressor, inputs, window)` refits `regressor` each step and predicts the next error with
an uncertainty.
- `inputs='none'`: estimate a constant level.
- `inputs='time'`: regress on the step number.
- `window=k`: use only the last k readings.

In the plots, the solid line is the mean and the dashed lines are ±2 SD.

In [ ]:
blr_all = Online(BayesianRidge(fit_intercept=False), inputs='none')

# TODO: the same BLR, but fit only on the last 20 readings
blr_recent = ...

plot({'BLR, all data': blr_all, 'BLR, last 20': blr_recent}, 'hover', seed=4);

**Q:** Why does "all data" adapt so slowly after step 100? What happens if you try windows of 5
and 100?

## 3. GP over time

The kernel has three parts:
- `RBF(length_scale)`: how long the error stays the same;
- `ConstantKernel`: how big it gets;
- `WhiteKernel`: the noise.

`fit` picks all three by maximizing the evidence.

In [ ]:
kernel = ConstantKernel(1.0, (1e-3, 1e2)) * RBF(20.0, (1.0, 1e3)) + WhiteKernel(1e-2, (1e-6, 1.0))

# TODO: an Online GP with this kernel, inputs='time', window=60
gp_time = ...

plot({'BLR, last 20': blr_recent, 'GP over time': gp_time}, 'hover', seed=4)
print('fitted kernel, output 0:', gp_time.fits[0].kernel_)

**Q:** What length scale did the evidence pick? Try `Matern(20.0, nu=0.5)` in place of `RBF`;
which fits jumps better?

## 4. Sensor noise

In [ ]:
# TODO: plot both models on 'hover', seed=4, with sensor=0.1

**Q:** Which model chases the noise? Do about 95% of the dots fall inside the dashed band?

## 5. Your turn

Start from the template below. `scores` checks a model on one world (fast):
- **error:** distance from the true error after the change (lower is better);
- **inside:** share of readings inside ±2 SD (good is about 0.95).

Also run it on a world with `change=False`: the error should stay near 0.

Ideas to try:
1. **Beat the GP over time** on `swing` with `sensor=0.1`.
2. **Detect the change:** compute P(world changed) at each step with Bayes' rule ("same world"
   vs "new world"). How many steps until you are sure? Any false alarms without a change?
3. **Calibrate before the change:** fit the kernel on steps 0-99 only, then freeze it
   (`GaussianProcessRegressor(kernel, optimizer=None)`). Why does it recover slowly?

In [ ]:
from labs.tracking.bench import stream, scores

# TODO: your model (another kernel, window, regressor, or your own class with reset/predict/update)
my_model = ...

world = stream('swing', seed=1, sensor=0.1)            # change=False for a world with no change
print(scores(my_model, world))
plot({'GP over time': gp_time, 'mine': my_model}, 'swing', seed=1, sensor=0.1);

Bring your best model and one plot to the share-out.

**Note:** this is a toy. Winners still get tested on the real environments later. Models with
`inputs='time+z'` can look great here and still fail in real control.